# Projekt Big Data — Detekcja ryzyka kredytowego (default)

**Silnik:** Apache Spark (PySpark) · **Streaming:** Kafka / Structured Streaming · **ML:** Spark MLlib  
**Dane:** UCI Default of Credit Card Clients (Yeh & Lien, 2009)  
**Środowisko:** Google Colab / lokalny Jupyter / Azure Databricks

### Tezy
1. **T1** — Historia opóźnień płatności (`PAY_*`) jest silniejszym predyktorem defaultu niż same limity.
2. **T2** — Wysoki wskaźnik wykorzystania limitu koreluje dodatnio z ryzykiem defaultu.
3. **T3** — Model MLlib osiąga **AUC > 0.70** na zbiorze testowym.
4. **T4** — Strumieniowe scorowanie (Kafka → Spark) umożliwia wczesne ostrzeganie.

## 0. Instalacja (Colab)

In [ ]:
# Odokumentuj w Colab — lokalnie pakiety mogą być już zainstalowane
import sys, subprocess
pkgs = ["pyspark==3.5.3", "pandas", "matplotlib", "seaborn", "scikit-learn", "kafka-python", "findspark"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("OK")

In [ ]:
from pathlib import Path
import json
import urllib.request

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

sns.set_theme(style="whitegrid")

# Ścieżki: w repozytorium lub lokalny katalog roboczy Colab
ROOT = Path("..").resolve() if Path("../data/raw").exists() else Path(".").resolve()
RAW = ROOT / "data" / "raw" / "UCI_Credit_Card.csv"
RAW.parent.mkdir(parents=True, exist_ok=True)
FIG = ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)

if not RAW.exists():
    url = "https://huggingface.co/datasets/scikit-learn/credit-card-clients/resolve/main/UCI_Credit_Card.csv"
    print("Pobieram dane UCI...")
    urllib.request.urlretrieve(url, RAW)

spark = (
    SparkSession.builder.appName("ColabCreditRisk")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Dane:", RAW)

## 1. Ekstrakcja i eksploracja danych

In [ ]:
df = spark.read.option("header", True).option("inferSchema", True).csv(str(RAW))
df = df.withColumnRenamed("default.payment.next.month", "default_payment_next_month")
print("Liczba rekordów:", df.count())
df.printSchema()
df.groupBy("default_payment_next_month").count().show()

In [ ]:
pdf = pd.read_csv(RAW).rename(columns={"default.payment.next.month": "default"})
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
pdf["default"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=["#2A6F97", "#C1121F"])
axes[0].set_title("Balans klas")
axes[0].set_xlabel("default")
rates = pdf.groupby("PAY_0")["default"].mean()
axes[1].plot(rates.index, rates.values, marker="o", color="#C1121F")
axes[1].set_title("T1: PAY_0 vs odsetek defaultów")
axes[1].set_xlabel("PAY_0")
axes[1].set_ylabel("P(default)")
plt.tight_layout()
plt.savefig(FIG / "colab_eda.png", dpi=140)
plt.show()

## 2. Transformacja (feature engineering) i zapis Parquet

In [ ]:
bill = [f"BILL_AMT{i}" for i in range(1, 7)]
pay = [f"PAY_AMT{i}" for i in range(1, 7)]
delays = ["PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6"]

feat = (
    df.withColumn("EDUCATION", F.when(F.col("EDUCATION").isin(0, 5, 6), 4).otherwise(F.col("EDUCATION")))
      .withColumn("MARRIAGE", F.when(F.col("MARRIAGE") == 0, 3).otherwise(F.col("MARRIAGE")))
      .withColumn("avg_bill", sum(F.col(c) for c in bill) / 6.0)
      .withColumn("avg_pay", sum(F.col(c) for c in pay) / 6.0)
      .withColumn("util_ratio", F.col("BILL_AMT1") / F.col("LIMIT_BAL"))
      .withColumn("pay_bill_ratio", F.when(F.col("avg_bill") != 0, F.col("avg_pay") / F.col("avg_bill")).otherwise(0.0))
      .withColumn("delay_count", sum(F.when(F.col(c) > 0, 1).otherwise(0) for c in delays).cast("double"))
      .filter(F.col("LIMIT_BAL") > 0)
)

out = ROOT / "data" / "processed" / "credit_features_colab.parquet"
feat.write.mode("overwrite").parquet(str(out))
feat.select("LIMIT_BAL", "AGE", "util_ratio", "delay_count", "default_payment_next_month").describe().show()
print("Zapisano:", out)

## 3. MLlib — RandomForest (klasyfikacja defaultu)

In [ ]:
FEATURE_COLS = [
    "LIMIT_BAL", "SEX", "EDUCATION", "MARRIAGE", "AGE",
    "PAY_0", "PAY_2", "PAY_3", "PAY_4", "PAY_5", "PAY_6",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3", "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3", "PAY_AMT4", "PAY_AMT5", "PAY_AMT6",
    "util_ratio", "avg_bill", "avg_pay", "pay_bill_ratio", "delay_count",
]
LABEL = "default_payment_next_month"

data = feat.select(*FEATURE_COLS, LABEL)
train, test = data.randomSplit([0.8, 0.2], seed=42)

pipeline = Pipeline(stages=[
    VectorAssembler(inputCols=FEATURE_COLS, outputCol="features_raw", handleInvalid="skip"),
    StandardScaler(inputCol="features_raw", outputCol="features"),
    RandomForestClassifier(featuresCol="features", labelCol=LABEL, numTrees=100, maxDepth=8, seed=42),
])

model = pipeline.fit(train)
pred = model.transform(test)

auc = BinaryClassificationEvaluator(labelCol=LABEL, rawPredictionCol="rawPrediction").evaluate(pred)
acc = MulticlassClassificationEvaluator(labelCol=LABEL, predictionCol="prediction", metricName="accuracy").evaluate(pred)
f1 = MulticlassClassificationEvaluator(labelCol=LABEL, predictionCol="prediction", metricName="f1").evaluate(pred)

print(f"AUC={auc:.4f}  Accuracy={acc:.4f}  F1={f1:.4f}")
print("TEZA T3:", "POTWIERDZONA" if auc > 0.70 else "NIEPOTWIERDZONA")
pred.groupBy(LABEL, "prediction").count().orderBy(LABEL, "prediction").show()

In [ ]:
rf = model.stages[-1]
imp = list(zip(FEATURE_COLS, rf.featureImportances.toArray().tolist()))
imp = sorted(imp, key=lambda x: x[1], reverse=True)[:10]
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh([i[0] for i in imp][::-1], [i[1] for i in imp][::-1], color="#2A6F97")
ax.set_title("Ważność cech — RandomForest (Spark MLlib)")
plt.tight_layout()
plt.savefig(FIG / "colab_importance.png", dpi=140)
plt.show()
print("Top cechy (T1):", [i[0] for i in imp[:5]])

## 4. Symulacja strumienia (Kafka-compatible JSON) + scorowanie

W Colab bez Dockera używamy pliku JSONL o schemacie wiadomości Kafki. Lokalnie: `docker compose up -d` + `python scripts/kafka_producer.py`.

In [ ]:
from datetime import datetime, timezone

stream_dir = ROOT / "data" / "stream_sim"
stream_dir.mkdir(parents=True, exist_ok=True)
stream_file = stream_dir / "transactions.jsonl"

sample = pdf.head(1500)
with open(stream_file, "w", encoding="utf-8") as fh:
    for i, row in sample.iterrows():
        event = {
            "event_id": f"txn-{i:06d}",
            "client_id": int(row["ID"]),
            "ts": datetime.now(timezone.utc).isoformat(),
            "limit_bal": float(row["LIMIT_BAL"]),
            "util_ratio": float(row["BILL_AMT1"] / row["LIMIT_BAL"]) if row["LIMIT_BAL"] else 0.0,
            "pay0": int(row["PAY_0"]),
            "age": int(row["AGE"]),
            "label_default": int(row["default"]),
        }
        score = (0.35 if event["util_ratio"] > 0.8 else 0) + (0.4 if event["pay0"] >= 2 else 0.2 if event["pay0"] == 1 else 0)
        event["risk_score"] = score
        event["risk_level"] = "HIGH" if score >= 0.55 else "MEDIUM" if score >= 0.30 else "LOW"
        fh.write(json.dumps(event) + "\n")

sdf = spark.read.json(str(stream_file))
print("Zdarzeń w strumieniu:", sdf.count())
sdf.groupBy("risk_level").count().show()
high = sdf.filter(F.col("risk_level") == "HIGH")
prec = high.filter(F.col("label_default") == 1).count() / max(high.count(), 1)
print(f"Precision HIGH risk (przybliżona walidacja T4): {prec:.3f}")

## 5. Wnioski decyzyjne

- **Scorecard kredytowy:** model RF + reguły strumieniowe mogą wspierać decyzje o limicie / windykacji.
- **Monitoring:** Kafka pozwala reagować na pogorszenie `PAY_*` i wzrost `util_ratio` w czasie zbliżonym do rzeczywistego.
- **Następne kroki (Databricks):** Delta Lake + Job scheduling + Model Registry; A/B test progów ryzyka.

```python
spark.stop()
```

In [ ]:
spark.stop()
print("Sesja Spark zamknięta.")